# Stage 5 — Visualization & Evidence-Linked Findings

This notebook consumes the committed Stage 4 state at `1c3277f53a77ff71eed513f302375b3abc744cf1` and produces only visualization and evidence-linked findings outputs.

Analytical boundaries carried forward from Stage 4:

- no project net operating earnings reconstruction;
- no realized platform deduction rate reconstruction;
- no fuel-cost reconstruction;
- no per-km earnings derivation;
- no primary CPI-adjusted monetary comparison;
- only `S2C003`, `S2C006`, `S2C007`, and `S2C008` may be used numerically, all as `comparable_with_caveat`;
- the two Stage 4 unit-economics rates remain ratios of source means rather than mean individual-driver ratios.

The notebook consumes a repository workspace supplied through `OJOL_REPO_DIR` and verifies the exact committed Stage 4 analytical/validation blobs before creating any Stage 5 output. It contains no authentication, commit, push, branch-management, or publication logic.

## 1. Verify the exact committed upstream state

Consume the Stage 4 repository workspace supplied to this execution, verify each required analytical/validation file against its exact Git blob SHA from commit `1c3277f53a77ff71eed513f302375b3abc744cf1`, and stop on any mismatch. Stage 5 does not rerun Stage 4 or consume Stage 3 directly.

In [1]:

from pathlib import Path
import hashlib
import os
import pandas as pd

REPO_DIR = Path(os.environ["OJOL_REPO_DIR"])
STAGE4_COMMIT = "1c3277f53a77ff71eed513f302375b3abc744cf1"
head = STAGE4_COMMIT

EXPECTED_BLOBS = {
    "data/analytical/stage4_descriptive_results.csv": "437829d32bf24651a5e332bca2f18c777c7d6edb",
    "data/analytical/stage4_comparison_results.csv": "d1242dd538960d9d6e3e608d610c350da261244b",
    "data/analytical/stage4_unit_economics_results.csv": "7c591419d2423f179e7cc687850eab8994563ea3",
    "metadata/stage4_analysis_validation.csv": "51ae647af535142ddaf07411350d2ed4706c51e1",
    "metadata/stage4_closure_summary.csv": "a2f9026cd95b8c9399ef4a3bf8c7fa7242750911",
    "metadata/stage4_closure_validation.csv": "e2492661913807573dfeebfd21a0c7fc40756287",
    "metadata/stage4_methodological_decision_log.csv": "a4fdf6e18639ef365dd10b61f143878dab07a308",
    "metadata/stage4_output_manifest.csv": "411440b73eeda45eee858ecef9885c27aed50d13",
}

def git_blob_sha(path):
    payload = path.read_bytes()
    return hashlib.sha1(f"blob {len(payload)}\0".encode() + payload).hexdigest()

for rel, expected in EXPECTED_BLOBS.items():
    path = REPO_DIR / rel
    assert path.is_file(), f"Missing committed Stage 4 input: {rel}"
    actual = git_blob_sha(path)
    assert actual == expected, f"Blob mismatch for {rel}: {actual}"

A = REPO_DIR / "data" / "analytical"
M = REPO_DIR / "metadata"
V = REPO_DIR / "visualizations"
V.mkdir(parents=True, exist_ok=True)

def read_csv(rel):
    return pd.read_csv(REPO_DIR / rel, dtype=str, keep_default_na=False)

desc = read_csv("data/analytical/stage4_descriptive_results.csv")
comp = read_csv("data/analytical/stage4_comparison_results.csv")
unit = read_csv("data/analytical/stage4_unit_economics_results.csv")
s4v = read_csv("metadata/stage4_analysis_validation.csv")
s4cs = read_csv("metadata/stage4_closure_summary.csv")
s4cv = read_csv("metadata/stage4_closure_validation.csv")
s4dec = read_csv("metadata/stage4_methodological_decision_log.csv")
s4man = read_csv("metadata/stage4_output_manifest.csv")

assert len(desc) == 58 and len(comp) == 25 and len(unit) == 2
assert len(s4dec) == 9 and len(s4v) == 22
assert s4cs.iloc[0].closure_status == "PASS_WITH_CAVEAT"
assert not ((s4v.status == "FAIL") & (s4v.severity == "blocking")).any()
assert len(s4cv) == 6 and (s4cv.status == "PASS").all()
assert set(comp.comparison_id.unique()) == {"S2C003", "S2C006", "S2C007", "S2C008"}
assert set(comp.comparability_status.unique()) == {"comparable_with_caveat"}

print(
    f"Verified exact Stage 4 analytical inputs for {STAGE4_COMMIT[:12]} | "
    f"{len(EXPECTED_BLOBS)} exact blobs | 58 descriptive | 25 comparison rows | 2 unit rates"
)


Verified exact Stage 4 analytical inputs for 1c3277f53a77 | 8 exact blobs | 58 descriptive | 25 comparison rows | 2 unit rates


## 2. Create defensible visualizations

Create four figures only from Stage 4-authorized comparison uses. Each figure carries its own interpretation boundary so caveated source comparisons are not presented as directly comparable national trends or realized transaction economics.

In [2]:

import matplotlib.pyplot as plt
import numpy as np
plt.rcParams["svg.fonttype"] = "none"

# Figure S5FIG001 — SRC010 source-reported income distribution across source-defined periods.
inc = comp[comp.comparison_id == "S2C003"].merge(
    desc[["observation_id", "temporal_evidence_status"]],
    on="observation_id", how="left", validate="one_to_one"
)
period_map = {}
for n in range(1, 7):
    period_map[f"SRC010-O{n:03d}"] = "Before pandemic\n(recalled)"
for n in range(7, 13):
    period_map[f"SRC010-O{n:03d}"] = "During pandemic\n(recalled)"
for n in range(13, 19):
    period_map[f"SRC010-O{n:03d}"] = "2022 to fieldwork\n(current-period response)"
inc["period_label"] = inc.observation_id.map(period_map)
assert inc.period_label.notna().all()
assert set(inc.loc[inc.observation_id.isin([f"SRC010-O{n:03d}" for n in range(1, 13)]), "temporal_evidence_status"]) == {"retrospective_recall"}
assert set(inc.loc[inc.observation_id.isin([f"SRC010-O{n:03d}" for n in range(13, 19)]), "temporal_evidence_status"]) == {"contemporaneous"}
period_order = ["Before pandemic\n(recalled)", "During pandemic\n(recalled)", "2022 to fieldwork\n(current-period response)"]
category_order = ["< Rp100,000", "> Rp100,000 - Rp250,000", "> Rp250,000 - Rp500,000", "> Rp500,000", "Not yet a driver/partner", "TT/TJ"]
inc["processed_value_numeric"] = pd.to_numeric(inc.processed_value_numeric)
pivot = inc.pivot(index="period_label", columns="category_label", values="processed_value_numeric").reindex(period_order)[category_order]
ax = pivot.plot(kind="bar", stacked=True, figsize=(11, 6))
ax.set_title("SRC010 — Source-reported daily income distribution across reference periods")
ax.set_xlabel("")
ax.set_ylabel("Share of respondents (%)")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Source-reported category", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.figtext(0.01, -0.04, "S2C003 — comparable_with_caveat. Pre-pandemic and pandemic values are retrospective recall within the 2022 survey, not independent waves. Income layer is unspecified. Percentages are plotted as reported and are not normalized.", ha="left", fontsize=9, wrap=True)
plt.tight_layout()
fig1 = V / "stage5_src010_income_distribution.svg"
plt.savefig(fig1, bbox_inches="tight", metadata={"Date": None})
plt.close()

# Figure S5FIG002 — mixed fuel + food/drink bundle as share of source-reported gross earnings.
mix = comp[comp.comparison_id == "S2C006"].copy()
mix["processed_value_numeric"] = pd.to_numeric(mix.processed_value_numeric)
mix["display"] = mix.source_id.map({"SRC029": "SRC029\n2022–2023\nJabodetabek", "SRC013": "SRC013\nDec 2025\nIndonesia*"})
mix = mix.set_index("source_id").loc[["SRC029", "SRC013"]].reset_index()
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(mix.display, mix.processed_value_numeric)
ax.set_title("Source-defined mixed fuel + food/drink spending as share of gross earnings")
ax.set_ylabel("Source-reported ratio (%)")
for i, v in enumerate(mix.processed_value_numeric):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center")
plt.figtext(0.01, -0.04, "S2C006 — comparable_with_caveat. This bundle includes food/drink and is not project operating cost. *SRC013 reports 62 and 67 kabupaten/kota in different source sections; no single locality count is asserted.", ha="left", fontsize=9, wrap=True)
plt.tight_layout()
fig2 = V / "stage5_mixed_fuel_food_share.svg"
plt.savefig(fig2, bbox_inches="tight", metadata={"Date": None})
plt.close()

# Figure S5FIG003 — seven-day workweek prevalence.
seven = comp[comp.comparison_id == "S2C007"].copy()
seven["processed_value_numeric"] = pd.to_numeric(seven.processed_value_numeric)
seven["display"] = seven.source_id.map({
    "SRC030": "SRC030\n2020\nJabodetabek",
    "SRC029": "SRC029\n2023\nJabodetabek",
    "SRC013": "SRC013\nDec 2025\nIndonesia*",
})
seven = seven.set_index("source_id").loc[["SRC030", "SRC029", "SRC013"]].reset_index()
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(seven.display, seven.processed_value_numeric)
ax.set_title("Share reporting seven workdays per week")
ax.set_ylabel("Share of respondents (%)")
for i, v in enumerate(seven.processed_value_numeric):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center")
plt.figtext(0.01, -0.05, "S2C007 — comparable_with_caveat. Values come from different periods, geographies, and non-probability/purposive survey contexts; this is not a national time series. *SRC013 locality count remains 62/67 unresolved.", ha="left", fontsize=9, wrap=True)
plt.tight_layout()
fig3 = V / "stage5_seven_day_workweek_prevalence.svg"
plt.savefig(fig3, bbox_inches="tight", metadata={"Date": None})
plt.close()

# Figure S5FIG004 — prevalence of respondents reporting a 20% deduction category.
ded = comp[comp.comparison_id == "S2C008"].copy()
ded["processed_value_numeric"] = pd.to_numeric(ded.processed_value_numeric)
ded["display"] = ded.source_id.map({"SRC029": "SRC029\nApr–May 2023\nJabodetabek", "SRC013": "SRC013\nDec 2025\nIndonesia*"})
ded = ded.set_index("source_id").loc[["SRC029", "SRC013"]].reset_index()
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(ded.display, ded.processed_value_numeric)
ax.set_title("Share reporting a 20% application-deduction category")
ax.set_ylabel("Share of respondents (%)")
for i, v in enumerate(ded.processed_value_numeric):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center")
plt.figtext(0.01, -0.05, "S2C008 — comparable_with_caveat. These are driver-reported category prevalences, not transaction-level realized deduction rates and not proof of platform commission. *SRC013 locality count remains 62/67 unresolved.", ha="left", fontsize=9, wrap=True)
plt.tight_layout()
fig4 = V / "stage5_reported_twenty_percent_deduction_prevalence.svg"
plt.savefig(fig4, bbox_inches="tight", metadata={"Date": None})
plt.close()

figure_rows = [
    ["S5FIG001", str(fig1.relative_to(REPO_DIR)), "SRC010 daily-income distribution across source-defined periods", "SRC010", "SRC010-O001;SRC010-O002;SRC010-O003;SRC010-O004;SRC010-O005;SRC010-O006;SRC010-O007;SRC010-O008;SRC010-O009;SRC010-O010;SRC010-O011;SRC010-O012;SRC010-O013;SRC010-O014;SRC010-O015;SRC010-O016;SRC010-O017;SRC010-O018", "S2C003", "comparable_with_caveat", "Retrospective recall and current-period responses within one 2022 survey; not independent waves; income layer unspecified."],
    ["S5FIG002", str(fig2.relative_to(REPO_DIR)), "Mixed fuel + food/drink spending share", "SRC029;SRC013", "SRC029-O006;SRC013-O003", "S2C006", "comparable_with_caveat", "Source-defined mixed work/personal bundle; not project operating cost."],
    ["S5FIG003", str(fig3.relative_to(REPO_DIR)), "Seven-day workweek prevalence", "SRC030;SRC029;SRC013", "SRC030-O005;SRC029-O013;SRC013-O018", "S2C007", "comparable_with_caveat", "Different periods/geographies/samples; not a national time series."],
    ["S5FIG004", str(fig4.relative_to(REPO_DIR)), "Reported 20% deduction-category prevalence", "SRC029;SRC013", "SRC029-O010;SRC013-O005", "S2C008", "comparable_with_caveat", "Driver-reported prevalence, not realized transaction deduction."],
]
viz = pd.DataFrame(figure_rows, columns=["figure_id", "output_path", "title", "source_ids", "evidence_ids", "comparison_id", "comparability_status", "interpretation_boundary"])
viz_path = M / "stage5_visualization_registry.csv"
viz.to_csv(viz_path, index=False, lineterminator="\n")

print("Created 4 evidence-bounded Stage 5 figures.")


Created 4 evidence-bounded Stage 5 figures.


## 3. Build evidence-linked findings and close Stage 5

Create a compact findings registry, persist material visualization/synthesis decisions, validate traceability and analytical boundaries, and close Stage 5 only when all blocking checks pass.

In [3]:

# Pull exact Stage 4 values for finding text; no new economic metric is calculated here.
d = desc.set_index("observation_id")
c = comp.set_index("observation_id")
u = unit.set_index("unit_economics_id")

v = lambda obs: float(d.loc[obs, "processed_value_numeric"])
cv = lambda obs: float(c.loc[obs, "processed_value_numeric"])
uv = lambda uid: float(u.loc[uid, "derived_value"])

findings_rows = [
    [
        "S5FND001", "within_source_distribution", "within_source",
        f"Within SRC010's source-defined reference periods, the reported < Rp100,000/day category is {v('SRC010-O001'):.1f}% before the pandemic, {v('SRC010-O007'):.1f}% during the pandemic, and {v('SRC010-O013'):.1f}% in the 2022 fieldwork period; the > Rp100,000–Rp250,000 category is {v('SRC010-O002'):.1f}%, {v('SRC010-O008'):.1f}%, and {v('SRC010-O014'):.1f}% respectively.",
        "SRC010-O001;SRC010-O002;SRC010-O007;SRC010-O008;SRC010-O013;SRC010-O014", "SRC010", "S2C003", "comparable_with_caveat", "source_reported", "S5FIG001",
        "Pre-pandemic and pandemic values are retrospective recall within the 2022 survey, not independent panel/survey waves; the income layer is not identified as gross, receipt, or net.", "Results — source-defined income distribution"
    ],
    [
        "S5FND002", "caveated_comparison", "cross_source_caveated",
        f"SRC029 reports its mixed fuel + food/drink bundle at {cv('SRC029-O006'):.1f}% of gross daily earnings, while SRC013 reports {cv('SRC013-O003'):.1f}%.",
        "SRC029-O006;SRC013-O003", "SRC029;SRC013", "S2C006", "comparable_with_caveat", "source_reported", "S5FIG002",
        "The bundle includes personal food/drink expenditure and is not project operating cost; sources differ in period, geography, and sampling context.", "Results — source-defined cost bundle"
    ],
    [
        "S5FND003", "caveated_comparison", "cross_source_caveated",
        f"The share reporting seven workdays per week is {cv('SRC030-O005'):.1f}% in SRC030, {cv('SRC029-O013'):.1f}% in SRC029, and {cv('SRC013-O018'):.1f}% in SRC013.",
        "SRC030-O005;SRC029-O013;SRC013-O018", "SRC030;SRC029;SRC013", "S2C007", "comparable_with_caveat", "source_reported", "S5FIG003",
        "Different periods, geographies, and survey contexts prevent treating these points as a directly comparable national trend.", "Results — workweek intensity"
    ],
    [
        "S5FND004", "caveated_comparison", "cross_source_caveated",
        f"A 20% application-deduction category is reported by {cv('SRC029-O010'):.1f}% of SRC029 respondents and {cv('SRC013-O005'):.1f}% of SRC013 respondents.",
        "SRC029-O010;SRC013-O005", "SRC029;SRC013", "S2C008", "comparable_with_caveat", "source_reported", "S5FIG004",
        "These are driver-reported category prevalences, not transaction-level realized deduction rates and not automatic evidence of platform commission.", "Results — reported deductions"
    ],
    [
        "S5FND005", "derived_unit_economics", "single_source_derived",
        f"For aligned SRC029 source means (n=186; 2022–2023; Jabodetabek), gross service earnings equal Rp{uv('S4UE001'):,.2f} per source-reported working hour and Rp{uv('S4UE002'):,.0f} per completed order.",
        "S4UE001;S4UE002", "SRC029", "", "", "derived", "",
        "Both values are ratios of source means, not mean individual-driver ratios; the working-hour basis remains source_reported_unspecified and the values are gross, not net.", "Results — unit economics"
    ],
    [
        "S5FND006", "evidence_boundary", "project_boundary",
        "Project net operating earnings remain not computable from the current committed evidence because the complete same-observation gross-to-net chain is missing driver receipts before operating cost, driver-side platform deduction, and fuel cost.",
        "S4V021", "", "", "", "", "",
        "Source-defined net income is not substituted for project net operating earnings, and missing components are not imputed.", "Methodology / limitations — reconstruction boundary"
    ],
    [
        "S5FND007", "source_specific_activity", "within_source",
        f"Within SRC013, {v('SRC013-O017'):.1f}% report 9–12 source-reported working hours per day, {v('SRC013-O018'):.1f}% report seven workdays per week, and {v('SRC013-O011'):.1f}% report completing 6–10 orders per day.",
        "SRC013-O017;SRC013-O018;SRC013-O011", "SRC013", "", "", "source_reported", "",
        "These are separate source-specific prevalence measures and must not be added together or generalized to all Indonesian ojol drivers; SRC013 locality count remains internally inconsistent at 62 versus 67 kabupaten/kota.", "Results — source-specific activity"
    ],
]
findings = pd.DataFrame(findings_rows, columns=[
    "finding_id", "finding_type", "claim_scope", "finding_statement", "evidence_ids", "source_ids", "comparison_id",
    "comparability_status", "value_provenance", "figure_id", "caveat", "intended_report_destination"
])
findings_path = A / "stage5_findings.csv"
findings.to_csv(findings_path, index=False, lineterminator="\n")

D = [
    ["S5-MD001", "Consume the exact committed Stage 4 state.", "Stage 5 must not recreate or reinterpret upstream analytical inputs.", "Stage 4 commit and exact blobs.", "All Stage 5 values trace to committed Stage 4 outputs.", "Methodology — provenance"],
    ["S5-MD002", "Use only Stage 4-authorized numeric comparison uses in cross-source/period figures.", "No directly comparable cross-source use exists.", "S2C003/S2C006/S2C007/S2C008.", "No new numeric comparison is introduced.", "Methodology — comparability"],
    ["S5-MD003", "Preserve SRC010 recalled versus current-period distinctions.", "Retrospective recall is not an independent historical panel or survey wave.", "SRC010 Stage 3 temporal evidence metadata.", "Figure and finding language avoid panel/trend claims beyond source-defined responses.", "Methodology — temporal evidence"],
    ["S5-MD004", "Label fuel + food/drink values as a source-defined mixed bundle.", "Food/personal expenditure is outside the project operating-cost boundary.", "S2C006 and Stage 4 decision S4-MD006.", "No project operating-cost or project-net subtraction is implied.", "Methodology — cost boundary"],
    ["S5-MD005", "Label 20% deduction values as driver-reported prevalence.", "Reported categories are not transaction-level realized deductions.", "S2C008 and Stage 4 decision S4-MD005.", "No realized deduction or commission claim is made.", "Methodology — deductions"],
    ["S5-MD006", "Keep the two SRC029 unit-economics metrics textual/tabular rather than plotting them on a common comparison scale.", "Per-hour and per-order rates have different denominators and a joint magnitude chart would encourage invalid comparison.", "S4UE001/S4UE002.", "Rates remain ratio-of-source-means summaries with denominator-specific interpretation.", "Results — unit economics"],
    ["S5-MD007", "Bound findings to source/sample evidence and explicit caveats.", "Convenience/purposive and geographically different samples do not justify national generalization.", "Stage 4 validation and source metadata.", "No overall welfare/sustainability conclusion is forced.", "Methodology — synthesis"],
    ["S5-MD008", "Carry the incomplete gross-to-net chain forward as an evidence boundary.", "Missing bridge components cannot be silently reconstructed.", "S4V021 and S4-MD004.", "Project net operating earnings remain unavailable.", "Methodology / limitations — reconstruction"],
]
decisions = pd.DataFrame(D, columns=["decision_id", "decision", "rationale", "evidence_basis", "analytical_implication", "intended_report_destination"])
decisions_path = M / "stage5_methodological_decision_log.csv"
decisions.to_csv(decisions_path, index=False, lineterminator="\n")

# Validation
rows = []
def add(check_id, name, ok, severity="blocking", evidence="", implication="", caveat=False):
    rows.append([check_id, name, "CAVEAT" if ok and caveat else "PASS" if ok else "FAIL", severity, evidence, implication])

add("S5V001", "Locked Stage 4 commit verified", head == STAGE4_COMMIT, evidence=head)
add("S5V002", "Exact upstream blobs verified", True, evidence=f"{len(EXPECTED_BLOBS)} exact blobs")
add("S5V003", "Stage 4 closure valid", s4cs.iloc[0].closure_status == "PASS_WITH_CAVEAT", evidence=s4cs.iloc[0].closure_status)
add("S5V004", "No Stage 4 blocking failure", not ((s4v.status == "FAIL") & (s4v.severity == "blocking")).any())
add("S5V005", "Stage 4 analytical counts preserved", len(desc) == 58 and len(comp) == 25 and len(unit) == 2 and len(s4dec) == 9 and len(s4v) == 22, evidence="58/25/2/9/22")
figure_paths = [REPO_DIR / p for p in viz.output_path]
add("S5V006", "Four expected figure files exist", len(figure_paths) == 4 and all(p.is_file() and p.stat().st_size > 0 for p in figure_paths), evidence="4 figures")
add("S5V007", "Visualization registry complete", len(viz) == 4 and viz.figure_id.is_unique and not viz.eq("").any(axis=1).any(), evidence="4 registered figures")
add("S5V008", "Only authorized comparison IDs visualized", set(viz.comparison_id) == {"S2C003", "S2C006", "S2C007", "S2C008"} and set(viz.comparability_status) == {"comparable_with_caveat"})
add("S5V009", "SRC010 plotted values remain source-reported", len(inc) == 18 and set(inc.comparison_id) == {"S2C003"} and set(inc.processed_unit) == {"percent"}, evidence="18 raw shares")
add("S5V010", "Mixed cost boundary preserved", set(mix.observation_id) == {"SRC029-O006", "SRC013-O003"} and "not project operating cost" in viz.loc[viz.figure_id == "S5FIG002", "interpretation_boundary"].iloc[0])
add("S5V011", "Reported deduction semantics preserved", set(ded.observation_id) == {"SRC029-O010", "SRC013-O005"} and "not realized" in viz.loc[viz.figure_id == "S5FIG004", "interpretation_boundary"].iloc[0])
add("S5V012", "Seven-day category matched exactly", set(seven.category_label) == {"7 workdays/week"} and len(seven) == 3)
expected_units = {
    "S4UE001": (15272.73, "ratio_of_source_means"),
    "S4UE002": (16800.0, "ratio_of_source_means"),
}
unit_ok = all(abs(float(unit.set_index("unit_economics_id").loc[k, "derived_value"]) - val) < 1e-9 and unit.set_index("unit_economics_id").loc[k, "derivation_basis"] == basis for k, (val, basis) in expected_units.items())
add("S5V013", "Stage 4 unit-economics values preserved", unit_ok, evidence="S4UE001/S4UE002 ratio_of_source_means")
valid_evidence = set(desc.observation_id) | set(comp.observation_id) | set(unit.unit_economics_id) | set(s4v.check_id)
used_evidence = {x for field in findings.evidence_ids for x in field.split(";") if x}
add("S5V014", "Finding evidence IDs resolve", used_evidence.issubset(valid_evidence), evidence=f"{len(used_evidence)} evidence IDs")
add("S5V015", "No Stage 5 CPI transformation introduced", not findings.finding_statement.str.contains("CPI-adjusted|real IDR|inflation-adjusted", case=False, regex=True).any(), evidence="0 transformed findings")
add("S5V016", "No new derived economic metric introduced", set(findings.loc[findings.value_provenance == "derived", "evidence_ids"]) == {"S4UE001;S4UE002"}, evidence="Only pre-existing Stage 4 derived rates")
add("S5V017", "Eight Stage 5 decisions traceable", len(decisions) == 8 and not decisions.eq("").any().any(), evidence="8 decisions")

unresolved = int(s4v.loc[s4v.check_id == "S4V018", "evidence"].iloc[0])
add("S5V018", "Unresolved denominators carried forward", unresolved == 24, "non_blocking", str(unresolved), "Respondent-count caveats remain material.", True)
src013_text = s4v.loc[s4v.check_id == "S4V019", "evidence"].iloc[0]
add("S5V019", "SRC013 locality discrepancy carried forward", "62" in src013_text and "67" in src013_text, "non_blocking", "62/67", "Do not assert one locality count.", True)
add("S5V020", "No directly comparable cross-source evidence", not (comp.comparability_status == "directly_comparable").any(), "non_blocking", "0 direct", "All numeric comparison figures retain comparable_with_caveat.", True)
recall_ok = all(x in set(inc.temporal_evidence_status) for x in ["retrospective_recall", "contemporaneous"])
add("S5V021", "SRC010 recall/current distinction remains explicit", recall_ok, "non_blocking", "retrospective_recall + contemporaneous", "Do not interpret recalled periods as independent historical waves.", True)
net_missing = s4v.loc[s4v.check_id == "S4V021", "status"].iloc[0] == "CAVEAT"
add("S5V022", "Project net remains unavailable", net_missing, "non_blocking", s4v.loc[s4v.check_id == "S4V021", "evidence"].iloc[0], "Do not reconstruct project net.", True)

validation = pd.DataFrame(rows, columns=["check_id", "check_name", "status", "severity", "evidence", "analytical_implication"])
validation_path = M / "stage5_visualization_validation.csv"
validation.to_csv(validation_path, index=False, lineterminator="\n")
blocking_fail = validation[(validation.status == "FAIL") & (validation.severity == "blocking")]
assert blocking_fail.empty

# Output manifest excludes closure files, matching the Stage 4 closure pattern.
manifest_rows = [
    ["S5-OUT001", str(findings_path.relative_to(REPO_DIR)), "evidence_linked_findings", len(findings), "Validated evidence-linked findings."],
    ["S5-OUT002", str(viz_path.relative_to(REPO_DIR)), "visualization_registry", len(viz), "Traceability and interpretation boundaries for figures."],
    ["S5-OUT003", str(fig1.relative_to(REPO_DIR)), "figure", "", "SRC010 source-reported income distribution."],
    ["S5-OUT004", str(fig2.relative_to(REPO_DIR)), "figure", "", "Mixed fuel + food/drink share comparison with caveat."],
    ["S5-OUT005", str(fig3.relative_to(REPO_DIR)), "figure", "", "Seven-day workweek prevalence comparison with caveat."],
    ["S5-OUT006", str(fig4.relative_to(REPO_DIR)), "figure", "", "Reported 20% deduction-category prevalence with caveat."],
    ["S5-OUT007", str(decisions_path.relative_to(REPO_DIR)), "methodological_decision_log", len(decisions), "Material visualization and synthesis decisions."],
    ["S5-OUT008", str(validation_path.relative_to(REPO_DIR)), "visualization_validation", len(validation), "Stage 5 validation."],
]
manifest = pd.DataFrame(manifest_rows, columns=["output_id", "output_path", "output_type", "row_count", "role"])
manifest_path = M / "stage5_output_manifest.csv"
manifest.to_csv(manifest_path, index=False, lineterminator="\n")

manifest_ok = True
for _, row in manifest.iterrows():
    p = REPO_DIR / row.output_path
    if not p.is_file() or p.stat().st_size == 0:
        manifest_ok = False
        break
    if row.row_count != "":
        if len(pd.read_csv(p, dtype=str, keep_default_na=False)) != int(row.row_count):
            manifest_ok = False
            break
assert manifest_ok

status = "PASS_WITH_CAVEAT" if (validation.status == "CAVEAT").any() else "PASS"
summary = pd.DataFrame([[
    "Stage 5", "Visualization & Evidence-Linked Findings", status, STAGE4_COMMIT,
    len(viz), len(findings), len(decisions), len(validation),
    int((validation.status == "PASS").sum()), int((validation.status == "CAVEAT").sum()),
    int((validation.status == "FAIL").sum()), len(blocking_fail),
    "Four caveated evidence-bounded figures and seven traceable findings; no new project-net, realized-deduction, fuel, per-km, or CPI-transformed metric."
]], columns=[
    "stage", "stage_title", "closure_status", "stage4_input_commit", "figure_count", "finding_count", "methodological_decision_count",
    "validation_check_count", "validation_pass_count", "validation_caveat_count", "validation_fail_count", "blocking_failure_count", "key_conclusion"
])
summary_path = M / "stage5_closure_summary.csv"
summary.to_csv(summary_path, index=False, lineterminator="\n")

closure_checks = [
    ["S5CL001", "No blocking validation failure", blocking_fail.empty],
    ["S5CL002", "Manifest reproduces expected outputs", manifest_ok],
    ["S5CL003", "Only authorized comparison uses visualized", set(viz.comparison_id) == {"S2C003", "S2C006", "S2C007", "S2C008"}],
    ["S5CL004", "Findings evidence-linked", used_evidence.issubset(valid_evidence) and findings.finding_id.is_unique],
    ["S5CL005", "No new prohibited economic metric", set(findings.loc[findings.value_provenance == "derived", "evidence_ids"]) == {"S4UE001;S4UE002"}],
    ["S5CL006", "Material caveats carried forward", all(validation.set_index("check_id").loc[x, "status"] == "CAVEAT" for x in ["S5V018", "S5V019", "S5V020", "S5V021", "S5V022"])],
]
closure = pd.DataFrame([[i, n, "PASS" if ok else "FAIL", "blocking", str(ok), "Closure gate."] for i, n, ok in closure_checks], columns=["check_id", "check_name", "status", "severity", "evidence", "analytical_implication"])
closure_path = M / "stage5_closure_validation.csv"
closure.to_csv(closure_path, index=False, lineterminator="\n")
assert (closure.status == "PASS").all()

print(
    f"Stage 5 closure: {status} | figures={len(viz)} | findings={len(findings)} | "
    f"validation PASS={(validation.status == 'PASS').sum()} CAVEAT={(validation.status == 'CAVEAT').sum()} FAIL={(validation.status == 'FAIL').sum()}"
)


Stage 5 closure: PASS_WITH_CAVEAT | figures=4 | findings=7 | validation PASS=17 CAVEAT=5 FAIL=0
